# Notebook 16 — Race Classification and Eligibility

## Bounded question

> What do the source race-classification and eligibility fields represent, how complete and internally consistent are they, how do their meanings vary by jurisdiction and race type, and which values can be safely preserved, parsed or derived?

## Initial governed scope

The source-field governance register places the following race-grain fields in `race_classification_and_conditions`:

- `race_name`
- `type`
- `class`
- `pattern`
- `rating_band`
- `age_band`
- `sex_rest`

`going` is deliberately excluded from this initial boundary because the governance register assigns it to the separate `race_conditions` family.

This notebook begins with source profiling only. It does not yet define parsers, canonical categories, regulatory equivalences, eligibility rules, or reusable implementation.

## Stage 1 — Establish storage and availability at runner and provisional-race grain

This first stage establishes the raw evidence boundary before interpreting any labels. It checks the SQLite storage classes, null/blank states, distinct raw values and within-race consistency for the seven governed fields.

The provisional race identity remains `date + course + off`. No assumption is made yet that any field has the same meaning across jurisdictions, racing codes or periods.

In [ ]:
# Input grain:
#   Governed source runner rows from SQLite table `data`, restricted by
#   DATA_ROW_PREDICATE = "rowid <> 1". Each source row represents one
#   runner record, while the fields profiled here are expected to describe
#   the containing race.
#
# Output grain:
#   1. One summary row per investigated source field.
#   2. One storage-class row per field and observed SQLite storage class.
#   3. One within-race consistency row per field.
#
# Purpose:
#   Establish raw storage, availability, distinctness and provisional-race
#   consistency before attempting semantic interpretation.
#
# Raw versus derived values:
#   Raw source values are not modified. `trimmed_text` is derived only to
#   distinguish null, empty/whitespace-only and populated states. The
#   provisional race key is derived from raw date + course + off values.
#
# Assumptions deliberately not made:
#   - no field is assumed to be complete merely because it is populated;
#   - no label is assumed to have a global or regulatory meaning;
#   - blanks are not converted to categories or null substitutes;
#   - repeated runner-row values are not treated as independent race facts;
#   - race_name text is not parsed for conditions at this stage;
#   - apparent contradictions are preserved for later investigation.
#
# Validation and failure behaviour:
#   The cell fails if the database or table is unavailable, if any governed
#   field is absent, if the governed source-row count differs from the
#   established 1,851,285 rows, or if the provisional race count differs
#   from the established 189,043 races.

from pathlib import Path
import sqlite3

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_DATABASE_PATH = (
    PROJECT_ROOT
    / "data/raw/form_2015-present/form_2015-present/raceform.db"
)
SOURCE_TABLE = "data"
DATA_ROW_PREDICATE = "rowid <> 1"
EXPECTED_RUNNER_ROWS = 1_851_285
EXPECTED_PROVISIONAL_RACES = 189_043

CLASSIFICATION_FIELDS = [
    "race_name",
    "type",
    "class",
    "pattern",
    "rating_band",
    "age_band",
    "sex_rest",
]
RACE_KEY_FIELDS = ["date", "course", "off"]

if not SOURCE_DATABASE_PATH.is_file():
    raise FileNotFoundError(f"Source database not found: {SOURCE_DATABASE_PATH}")

connection_uri = f"file:{SOURCE_DATABASE_PATH}?mode=ro"
with sqlite3.connect(connection_uri, uri=True) as connection:
    table_info = pd.read_sql_query(
        f"PRAGMA table_info({SOURCE_TABLE})",
        connection,
    )

    observed_columns = set(table_info["name"])
    required_columns = set(RACE_KEY_FIELDS + CLASSIFICATION_FIELDS)
    missing_columns = sorted(required_columns - observed_columns)
    if missing_columns:
        raise AssertionError(f"Required source columns are missing: {missing_columns}")

    population = pd.read_sql_query(
        f"""
        SELECT
            COUNT(*) AS runner_rows,
            COUNT(DISTINCT date || char(31) || course || char(31) || off)
                AS provisional_races
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        """,
        connection,
    )

    runner_rows = int(population.loc[0, "runner_rows"])
    provisional_races = int(population.loc[0, "provisional_races"])
    assert runner_rows == EXPECTED_RUNNER_ROWS, (
        f"Governed runner population changed: {runner_rows:,} != "
        f"{EXPECTED_RUNNER_ROWS:,}"
    )
    assert provisional_races == EXPECTED_PROVISIONAL_RACES, (
        f"Provisional race population changed: {provisional_races:,} != "
        f"{EXPECTED_PROVISIONAL_RACES:,}"
    )

    field_summary_parts = []
    storage_summary_parts = []
    race_consistency_parts = []

    for field in CLASSIFICATION_FIELDS:
        field_summary_parts.append(
            pd.read_sql_query(
                f"""
                SELECT
                    '{field}' AS source_field,
                    COUNT(*) AS runner_rows,
                    SUM(CASE WHEN [{field}] IS NULL THEN 1 ELSE 0 END)
                        AS null_rows,
                    SUM(
                        CASE
                            WHEN [{field}] IS NOT NULL
                             AND TRIM(CAST([{field}] AS TEXT)) = ''
                            THEN 1 ELSE 0
                        END
                    ) AS blank_rows,
                    SUM(
                        CASE
                            WHEN [{field}] IS NOT NULL
                             AND TRIM(CAST([{field}] AS TEXT)) <> ''
                            THEN 1 ELSE 0
                        END
                    ) AS populated_rows,
                    COUNT(
                        DISTINCT CASE
                            WHEN [{field}] IS NOT NULL
                             AND TRIM(CAST([{field}] AS TEXT)) <> ''
                            THEN CAST([{field}] AS TEXT)
                        END
                    ) AS distinct_populated_raw_values
                FROM {SOURCE_TABLE}
                WHERE {DATA_ROW_PREDICATE}
                """,
                connection,
            )
        )

        storage_summary_parts.append(
            pd.read_sql_query(
                f"""
                SELECT
                    '{field}' AS source_field,
                    typeof([{field}]) AS sqlite_storage_class,
                    COUNT(*) AS runner_rows
                FROM {SOURCE_TABLE}
                WHERE {DATA_ROW_PREDICATE}
                GROUP BY typeof([{field}])
                ORDER BY runner_rows DESC, sqlite_storage_class
                """,
                connection,
            )
        )

        race_consistency_parts.append(
            pd.read_sql_query(
                f"""
                WITH race_values AS (
                    SELECT
                        date,
                        course,
                        off,
                        COUNT(DISTINCT
                            CASE
                                WHEN [{field}] IS NULL THEN '<NULL>'
                                ELSE typeof([{field}]) || ':' || CAST([{field}] AS TEXT)
                            END
                        ) AS distinct_raw_states
                    FROM {SOURCE_TABLE}
                    WHERE {DATA_ROW_PREDICATE}
                    GROUP BY date, course, off
                )
                SELECT
                    '{field}' AS source_field,
                    COUNT(*) AS provisional_races,
                    SUM(CASE WHEN distinct_raw_states = 1 THEN 1 ELSE 0 END)
                        AS internally_consistent_races,
                    SUM(CASE WHEN distinct_raw_states > 1 THEN 1 ELSE 0 END)
                        AS internally_mixed_races,
                    MAX(distinct_raw_states) AS maximum_states_within_race
                FROM race_values
                """,
                connection,
            )
        )

field_summary = pd.concat(field_summary_parts, ignore_index=True)
storage_summary = pd.concat(storage_summary_parts, ignore_index=True)
race_consistency = pd.concat(race_consistency_parts, ignore_index=True)

field_summary["populated_runner_pct"] = (
    field_summary["populated_rows"] / field_summary["runner_rows"] * 100
).round(3)
race_consistency["internally_mixed_race_pct"] = (
    race_consistency["internally_mixed_races"]
    / race_consistency["provisional_races"]
    * 100
).round(4)

print("Governed source population")
display(population)

print("Field availability and raw distinctness")
display(field_summary)

print("Observed SQLite storage classes")
display(storage_summary)

print("Within-provisional-race raw-state consistency")
display(race_consistency)
